
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img
    src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
    alt="Databricks Learning"
  >
</div>


# 문서 파싱과 청킹

## 서론

**Retrieval Augmented Generation(RAG)** 애플리케이션의 효과는 데이터 검색의 품질에 근본적으로 제약을 받습니다. **임베딩 생성** 전에 PDF, HTML 파일, 이미지와 같은 원시 **비정형 데이터**를 수집, 저장하고, **대형 언어 모델(LLM)** 이 해석할 수 있는 형식으로 변환해야 합니다. 이 수업은 **Databricks Intelligence Platform** 내 데이터 준비 단계에 초점을 맞추며, 특히 저장을 위한 **Delta Lake**와 거버넌스를 위한 **Unity Catalog**를 활용합니다. 우리는 이진 파일에서 구조화된 텍스트를 추출하는 네이티브 **`ai_parse_document` AI 함수**를 살펴보고, **텍스트 청킹**에 대한 중요한 전략을 탐구할 것입니다. 이는 기본적인 **고정 크기 분할**을 넘어 **컨텍스트 인식 방법**으로 나아갈 것입니다.

## 수업 목표

* RAG의 비정형 데이터 저장에서 Delta Lake 및 Unity Catalog Volumes의 역할을 설명하세요.  
* Databricks AI 함수, 특히 `ai_parse_document` 및 v2.0 그림 설명과 같은 기능을 설명하세요.  
* Autoloader 및 Spark Declarative Pipelines(SDP)를 사용하여 효율적인 섭취 패턴을 식별하세요.  
* 고정 크기, 재귀, 임베딩 기반의 의미론적 청킹 전략을 비교하고 대조하세요.  
* Databricks에서 파싱과 청킹에 사용되는 표준 도구와 프레임워크 매핑하세요.

## A. 데이터 저장 및 처리 아키텍처

RAG 아키텍처에서는 데이터 저장이 원시 소스 파일과 처리된 구조화된 텍스트를 모두 수용해야 합니다. **Delta Lake**는 통합 데이터 관리 계층 역할을 하여 모든 데이터 유형에 대해 ACID 트랜잭션과 버전 관리를 제공합니다. Delta 테이블은 구조화된 데이터에 최적화되어 있지만, RAG Workflows는 일반적으로 PDF와 같은 비구조화 파일로 시작합니다.

**Unity Catalog 볼륨**은 이러한 비표 파일에 대한 거버넌스 계층을 제공합니다. 볼륨은 테이블과 모델에 적용되는 동일한 통합 권한 모델을 사용하여 원시 파일 접근을 관리할 수 있게 해줍니다. 원시 문서를 Volumes에 저장하고 처리된 텍스트를 Delta Tables에 저장함으로써 원본 파일에서 검색에 사용되는 청크 텍스트까지의 완전한 계보를 유지할 수 있습니다.

**참고:** 볼륨은 "원시" 파일(브론즈), Delta 테이블은 "파싱되고 청크된" 텍스트(실버/골드)를 저장합니다.

아래는 데이터 수집 및 처리 개요 워크플로입니다. 일반적인 사용 사례에서—그리고 이 모듈에서—우리는 워크플로를 따라가며, 첫 세 단계에 집중합니다:

1. **데이터 수집 및 전처리:** Unity Catalog 볼륨에서 파일을 읽고 AI 함수로 파싱합니다.
1. **데이터 저장:** 파싱된 문서를 Delta Lake로 저장하고 필요한 거버넌스 제어를 적용하세요. (거버넌스는 이 모듈에서 자세히 다루지 않습니다.)
1. **청크:** 데이터를 임베딩 생성에 적합한 청크로 나누세요.

<!-- <img src="../Includes/images/02-process-diagram.png" alt="Data storage, ingestion and processing workflow" /> -->

![02-process-diagram](https://files.training.databricks.com/binder/prod_main/building-retrieval-agents-on-databricks-ko_kr-1.0.1/images/02-process-diagram.png)

*그림 1. 이 도표는 데이터 수집, 처리 및 임베딩 생성 워크플로의 5가지 주요 단계를 보여줍니다.* 


## B. AI 기능을 이용한 문서 처리

문서 처리는 특히 문서가 주요 지식 소스일 때 검색 에이전트의 고품질 지식 기반을 구축하는 데 필수적입니다. 실제 문서는 종종 복잡한 구조를 가지고 있습니다—예를 들어, 이러한 문제를 해결하기 위해 우리는 다양한 문서 형식에서 정보를 해석하고 추출하도록 설계된 대형 언어 모델(LLM)과 OCR 지원 LLM과 같은 첨단 모델을 활용합니다. Databricks는 이 과정을 간소화하기 위해 네이티브 AI 함수를 제공합니다. 특히, **`ai_parse_document` AI 함수**는 PDF와 이미지의 강력한 파싱을 가능하게 하며, 원시 파일에서 구조화된 콘텐츠와 레이아웃 정보를 직접 추출할 수 있습니다.

### B1. 문서 처리 과제

실제 문서의 파싱은 단순한 텍스트가 거의 아니기 때문에 복잡합니다. 문서에는 종종 **이미지**, **다중 열 레이아웃**, **표**, **도표**, **헤더**, **하위 헤더**, **페이지 번호**가 혼합되어 있습니다. 이 정보를 적절히 추출하면서 의미적 의미를 유지하는 데는 여러 도전 과제가 있습니다:

* **계층적 정보:** 차트와 다이어그램은 종종 보존해야 할 계층적 관계를 전달합니다.  
* **순서 보존:** 다열 문서에서는 읽기 순서가 매우 중요합니다; 순진한 파싱은 merge 열을 잘못 설정할 수 있습니다.  
* **맥락적 무결성:** 이미지(차트나 제품 사진과 같은)는 관련 텍스트 설명과 연관되어 있어야 합니다.



<!-- <img src="../Includes/images/02-complex-page-structure-example.png" alt="An example page showing complex page layout" /> -->
![02-complex-page-structure-example](https://files.training.databricks.com/binder/prod_main/building-retrieval-agents-on-databricks-ko_kr-1.0.1/images/02-complex-page-structure-example.png)

*그림 2. 이 이미지는 복잡한 페이지 레이아웃을 가진 예시 페이지 레이아웃을 보여줍니다* 




### B2. 구문 분석용 LLM과 OCR

이러한 문제를 해결하기 위해 현대적 접근법은 **대형 언어 모델(LLM)** 과 **광학 문자 인식(OCR)** 모델을 활용합니다. 전통적인 텍스트 파서와 달리, 이 모델들은 문서 레이아웃을 "볼 수 있습니다." OCR 모델은 이미지 내 텍스트를 식별할 수 있으며, 멀티모달 LLM은 그것 위 이미지에 캡션이 속하거나 테이블이 여러 페이지에 걸쳐 있음을 이해하여 요소의 공간적 배열을 해석할 수 있습니다.

### B3. `ai_parse_document` 사용

Databricks은 **AI 함수**를 통해 이 과정을 단순화하여, 개발자가 간단한 SQL 또는 Python 함수 호출을 통해 고급 AI 모델을 데이터에 직접 적용할 수 있게 합니다. 이로 인해 별도의 모델 추론 인프라를 관리할 필요가 없어집니다. 이 함수들은 서버리스을 실행하며, 수백만 행을 자동으로 처리할 수 있도록 확장되고, Unity Catalog 내에서 직접 관리되는 데이터에 대해 작동합니다.

**`ai_parse_document` AI 함수**는 이 작업에 있어 가장 중요한 Databricks 도구입니다. 최신 생성형 AI 모델을 활용하여 비정형 문서(예: PDF 및 이미지)에서 구조화된 콘텐츠를 추출하고, 그 결과를 구조화된 JSON 객체(VARIANT 유형)로 반환합니다.

**주요 기능 (스키마 v2.0):**

* **레이아웃 인식:** 문서 내용과 레이아웃 정보를 분리합니다.  
* **그림 설명:** PDF 내 차트와 이미지에 대한 텍스트 설명을 자동으로 생성할 수 있어 LLM이 시각적 데이터를 접근할 수 있도록 합니다.  
* **경계 박스:** 텍스트 요소에 대한 좌표(bbox)를 반환하여 UI에서 출처를 강조하는 데 유용합니다.

**예제 구현:**

```sql
-- 이진 PDF 데이터에서 문서 레이아웃과 내용을 추출합니다  
SELECT ai_parse_document(content) as parsed_document  
FROM read_files(  
  '/Volumes/path/to/pdfs/',  
  format => 'binaryFile'  
)
```

## C. 데이터 정제 및 변환

문서를 파싱한 후에는 파싱된 내용을 정리하고 그것을 목표에 맞는 형식으로 변환해야 합니다.



### C1. 소음 감소

텍스트를 청크화하기 전에 클리닝을 통해 검색 품질을 저하시키는 아티팩트를 제거해야 합니다. 원시 추출에는 종종 헤더, 푸터, 페이지 번호가 포함되어 텍스트의 의미적 흐름을 방해할 수 있습니다. 파싱 단계의 출력에는 클린 로직을 적용해야 합니다. HTML 데이터의 경우, 과도한 서식 태그은 모델을 혼란스럽게 할 수 있지만, `ai_parse_document` HTML 형식의 테이블을 지능적으로 추출하여 웹 페이지를 파싱하고 표 형식의 데이터를 해석 가능하게 유지하는 데 필수적인 구조를 유지할 수 있습니다.

### C2. 메타데이터 주입

효과적인 RAG 시스템은 벡터 유사성 검색 *전에* 메타데이터를 이용해 검색 결과를 필터링합니다. 변환 과정에서 문서 제목, 저자 이름, 생성 날짜와 같은 메타데이터를 추출하고 연관시키는 것이 매우 중요합니다. 이 메타데이터가 파일 속성에 없을 경우, **ai_extract**와 같은 함수를 사용해 비구조화 텍스트에서 구조화된 필드(예: "송장 날짜"나 "계약 유형")를 식별하고 Pull할 수 있습니다.

## D. 청킹 전략
청킹은 긴 문서를 더 작고 관리 가능한 구간으로 나누는 과정입니다. 이 단계는 임베딩 모델이 컨텍스트 윈도우 제한이 있어 정확한 정보를 얻으려면 세분화된 검색 결과가 필요하기 때문에 필수적입니다.

또 다른 중요한 고려사항은 문맥 크기와 언어 모델 성능 간의 관계입니다. "Lost in the Middle" 현상은 LLM이 큰 맥락 창 깊숙이 숨겨진 정보를 간과할 때 발생합니다. 따라서 중요한 세부 사항을 놓치지 않도록 더 작고 관련성 높은 청크를 만드는 것이 선호됩니다.

핵심 질문은 문서를 어떻게 가장 잘 분리할까 하는 점입니다. 여러 가지 청킹 방법이 있으며, 이 섹션에서는 가장 일반적이고 효과적인 접근법을 살펴보겠습니다.

**팁**: 청크 크기와 분배기를 기반으로 청킹을 시각화하려면 [ChunkViz](https://chunkviz.up.railway.app/)를 확인해 보세요.

### D1. 고정 크기 vs. 재귀적 청킹

* **고정 크기 청크(레거시/베이스라인):** 텍스트를 하드 문자 수나 토큰 수(예: 500 tokens)에 따라 나눕니다. 그것은 계산 비용이 저렴하지만 문장이나 단락을 반으로 나누어 맥락을 파괴하는 경우가 많습니다.  

* **의미 청킹(권장 표준):** 임의의 문자 분할과 달리, 이 접근법은 **문장**, **단락**, **문서 섹션**과 같은 의미 있는 언어 경계에 따라 텍스트를 구분합니다. 문서의 논리적 구조를 존중함으로써 이는 정보의 의미적 완전성을 유지합니다. 더 나아가, 의미 청킹은 관련 **메타데이터**, **태그**, **제목**을 청크에 직접 주입하는 경우가 많아, 작은 텍스트 구간도 검색 시 더 넓은 맥락을 유지하도록 보장합니다.


<!-- <img src="../Includes/images/02-chunking-methods.png" alt="Chunking methods visualized"/> -->

![02-chunking-methods](https://files.training.databricks.com/binder/prod_main/building-retrieval-agents-on-databricks-ko_kr-1.0.1/images/02-chunking-methods.png)

*그림 3. 고정 크기 청킹과 의미론적 청킹의 시각적 표현입니다.*

### D2. 고급 청킹 전략

검색 성능을 극대화하고 의미적 일관성을 보장하기 위해서는 복잡한 문서를 처리하고 경계를 넘어 맥락을 보존하는 더 정교한 전략이 필요합니다.

* **청크 오버랩:** 이 기법은 연속된 청크 간의 중복 정도(예: 10-20%)를 정의합니다. 다음 문단 시작 부분에 텍스트의 일부를 반복함으로써, 문단 간에 맥락 정보가 상실되지 않도록 하여 문장이나 아이디어가 갑작스럽게 끊어지는 것을 방지합니다.  

* **임베딩 기반 의미 청킹:** 이 방법은 임베딩 모델을 사용해 끊김점을 결정하는 더 발전된 방법입니다. 그것은 연속된 문장 간의 의미적 유사성을 계산하며, 주제가 크게 변할 때(즉, 유사도가 임계값 이하로 떨어질 때) 청크를 "분할"합니다. 이로 인해 각 청크가 독특하고 일관된 개념을 나타낼 수 있습니다.  

* **Windowed 요약:** 각 청크에 이전 몇 개의 청크에 대한 'Windowed 요약'을 포함하는 '맥락 풍부화' 청킹 방법입니다. 현재 텍스트만 보는 대신, 모델은 이전 내용을 요약하여 전체 역사를 내재하는 비용 없이 더 넓은 맥락을 제공합니다.

<!-- <img src="../Includes/images/02-advanced-chunking-methods.png" alt="Advanced chunking methods visualized"/> -->

![02-advanced-chunking-methods](https://files.training.databricks.com/binder/prod_main/building-retrieval-agents-on-databricks-ko_kr-1.0.1/images/02-advanced-chunking-methods.png)

*그림 4. 고급 청킹 기법의 시각적 표현입니다. 각 색깔은 하나의 덩어리를 나타냅니다.* 


### D3. 모델 임베딩 고려사항

임베딩 모델은 이후 모듈에서 자세히 다루지만, 기술적 제약은 *지금* 청킹 단계에서 고려해야 합니다.

* **컨텍스트 Window 제한:** 모든 임베딩 모델에는 최대 토큰 한도가 있습니다(예: 512, 8192 tokens). 텍스트 청크가 이 한계를 초과하면 모델은 단순히 텍스트를 **잘라내고** 한계를 넘는 내용은 무시합니다. 이로 인해 벡터 표현이 불완전해지고 데이터가 손실됩니다. 따라서 최대 청크 크기는 항상 임베딩 모델의 컨텍스트 Window 제한보다 안전하게 아래에 있어야 합니다.  

* **그라뉼래리티 vs. 컨텍스트:** 더 큰 컨텍스트 Window는 더 큰 청크를 허용하여 더 많은 맥락을 담을 수 있지만 특정 세부사항을 희석시킬 수 있습니다. 더 작은 창은 더 작은 덩어리를 강제로 만들며, 이는 더 정밀하지만 주변 맥락이 부족할 수 있습니다. 청크 크기 선택은 직접적인 절충이며, 향후 사용하려는 특정 임베딩 모델의 성능에 맞춰야 합니다.

## E. 청크용 도구 및 프레임워크

Databricks에서의 문서 처리는 네이티브 AI 함수와 LangChain 같은 주요 오픈 소스 라이브러리를 결합하여 견고한 다단계 pipeline을 만듭니다. 이 workflow는 순차적 파싱과 청킹을 통해 원시 파일을 임베드 가능한 텍스트로 변환합니다.

1. **파싱(추출):** 첫 단계는 원시 파일을 파싱하고 레이아웃 정보와 함께 깨끗한 텍스트를 추출하는 것입니다.  
   * **`ai_parse_document` (네이티브):** 이 추천 도구는 표준 문서(PDF, 이미지)를 효율적으로 처리하며, OCR 및 레이아웃 분석을 서버리스로 수행합니다. 이는 하위 작업에 준비가 된 구조화된 텍스트를 반환합니다.  
1. **청크(분할):** 추출 후에는 텍스트를 더 작고 관리 가능한 청크로 나누어야 합니다.  
   * **LangChain:** LangChain와 같은 라이브러리는 파싱된 텍스트에 대해 고급 분할 논리(예: `RecursiveCharacterTextSplitter` )를 제공합니다. LangChain의 텍스트 분할기 제품군은 다양한 형식과 전략을 지원하여 그것이 청크 처리의 업계 표준이 되었습니다.  
   * **사용자 정의 함수:** 개발자는 `ai_parse_document`의 출력에 특정 마크다운 헤더로 텍스트를 분할하는 등 특수한 분할 로직을 적용하기 위해 사용자 정의 파이썬 함수(UDF)를 구현할 수도 있습니다.

## F. 요약

Databricks에 대한 RAG 데이터 준비는 신뢰할 수 있는 파이프라인 인게스팅, 파싱, 변환을 포함합니다. 원본 파일은 먼저 Unity Catalog 볼륨에 인제스트됩니다. 그 후, LLM과 OCR을 활용하여 PDF 같은 복잡한 문서에서 깨끗한 텍스트와 레이아웃 정보를 추출하는 네이티브 **`ai_parse_document` AI 함수**를 사용하여 파싱됩니다. 마지막으로, 이 텍스트는 **재귀 문자 분할** 또는 **임베딩 기반 의미 청킹**과 같은 고급 방법을 사용하여 전략적으로 청크화되어 검색 시스템에서 임베딩 모델 제약 조건을 존중하면서 정확하고 맥락이 풍부한 정보를 얻을 수 있도록 합니다.

**핵심 내용:**

1. **통합 거버넌스:** 원시 파일을 Unity Catalog 볼륨에, 처리된 청크는 Delta 테이블에 저장하여 완전한 데이터 계보와 보안을 유지합니다.  
2. **순차 처리:** 문서 작성은 두 단계로 이루어집니다: 첫째, **`ai_parse_document`** 를 강력한 추출(OCR/레이아웃)에 사용하고, 둘째, 논리적 분할을 위해 LangChain 같은 라이브러리를 사용하세요.  
3. **고급 청크화:** 단순한 고정 크기 분할을 넘어서야 합니다. 맥락 손실을 방지하고 검색 정확도를 높이기 위해 **의미 전략**, **중복**, **상위 문서 검색**을 채택하세요.

&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>